In [ ]:
## Stochastic Simulation - Day 4

In [ ]:
# Exercise 5 - Part 1

Estimate the integral 

\begin{equation}
\int_0^1 e^x \: dx
\end{equation}

by simulating (for example 100 samples) using the crude Monte Carlo estimator. 

Present results as both a point estimate and a confidence interval. 

In [ ]:
import numpy as np

N = 100

# Generating 100 uniform distributed random numbers
random_numbers = np.random.uniform(0, 1, N)
# Defining X_i = e^U_i
X = np.exp(random_numbers)
# Crude Monte Carlo estmator 
estimator = 1/N * np.sum(X)
# Standard deviation of the estimator
std = np.std(X, ddof=1)
# Confidence interval of the estimator
ci_low = estimator - 1.96 * std / np.sqrt(N)
ci_high = estimator + 1.96 * std / np.sqrt(N)

print("Point estimate of crude Monte Carlo estimator:", estimator)
print("Standard deviation of crude Monte Carlo estimator:", std)
print("Confidence interval of crude Monte Carlo estimator: [", ci_low, ", ", ci_high, "]")

# Exercise 5 - Part 2: Estimate the same integral now using antithetic variables and comparable computational resources.

# Generating the antithetic variable 
Y = (X + np.exp(1 - X)) / 2
# Antithetic estimator
antithetic_estimator = 1/N * np.sum(Y)
# Standard deviation of the antithetic estimator
antithetic_std = np.std(Y, ddof=1)
# Confidence interval of the antithetic estimator
antithetic_ci_low = antithetic_estimator - 1.96 * antithetic_std / np.sqrt(N)
antithetic_ci_high = antithetic_estimator + 1.96 * antithetic_std / np.sqrt(N)

print("Point estimate of antithetic estimator:", antithetic_estimator)
print("Standard deviation of antithetic estimator:", antithetic_std)
print("Confidence interval of antithetic estimator: [", antithetic_ci_low, ", ", antithetic_ci_high, "]")

# Exercise 5 - Part 3: Estimate the same integral using a control variate and comparable computational resources.

# Generating the control variate
cov_matrix = np.cov(X, random_numbers)
cov_XU = cov_matrix[0, 1]
c = -cov_XU / np.var(random_numbers)
y = X + c*(random_numbers - 0.5) # using the true mean
# Control variate estimator
control_variate_estimator = np.mean(y)
# Standard deviation of the control variate estimator
control_variate_std = np.std(y, ddof=1)
# Confidence interval of the control variate estimator
control_variate_ci_low = control_variate_estimator - 1.96 * control_variate_std / np.sqrt(N)
control_variate_ci_high = control_variate_estimator + 1.96 * control_variate_std / np.sqrt(N)

print("Point estimate of control variate estimator:", control_variate_estimator)
print("Standard deviation of control variate estimator:", control_variate_std)
print("Confidence interval of control variate estimator: [", control_variate_ci_low, ", ", control_variate_ci_high, "]")

# Exercise 5 - Part 4: Estimate the same integral using stratified sampling and comparable computational resources.

# Number of strata
num_strata = 10
# One sample per stratum
U = np.zeros(num_strata)

for i in range(num_strata):
    # Sample uniformly inside the i'th stratum
    U[i] = np.random.uniform(i / num_strata, (i + 1) / num_strata)

Yy = np.exp(U)
# Stratified estimator
stratified_estimator = np.mean(Yy)
# Standard deviation of the stratified estimator
stratified_std = np.std(Yy, ddof=1)
# Confidence interval of the stratified estimator
stratified_ci_low = stratified_estimator - 1.96 * stratified_std / np.sqrt(N)
stratified_ci_high = stratified_estimator + 1.96 * stratified_std / np.sqrt(N)

print("Point estimate of stratified estimator:", stratified_estimator)
print("Standard deviation of stratified estimator:", stratified_std)
print("Confidence interval of stratified estimator: [", stratified_ci_low, ", ", stratified_ci_high, "]")


Point estimate of crude Monte Carlo estimator: 1.6679623898825082
Standard deviation of crude Monte Carlo estimator: 0.5025141546865117
Confidence interval of crude Monte Carlo estimator: [ 1.569469615563952 ,  1.7664551642010644 ]
Point estimate of antithetic estimator: 1.121159886870679
Standard deviation of antithetic estimator: 0.13023654916368246
Confidence interval of antithetic estimator: [ 1.095633523234597 ,  1.1466862505067608 ]
Point estimate of control variate estimator: 1.7231108258544807
Standard deviation of control variate estimator: 0.0628094399866098
Confidence interval of control variate estimator: [ 1.7108001756171052 ,  1.7354214760918563 ]
Point estimate of stratified estimator: 1.7213055192091276
Standard deviation of stratified estimator: 0.531969621375576
Confidence interval of stratified estimator: [ 1.6170394734195146 ,  1.8255715649987405 ]


In [24]:
# Exercise 5 - Part 5: Use control variates to reduce the variance of the estimator in Exercise 4 (Poisson arrivals)

# We use the total arrival time (the sum of all interarrival times) as a control variate because it is correlated 
# with the blocked fraction and its expected value is known, which allows us to reduce the variance of the blocked‑fraction estimator.

import numpy as np

def block_system_poisson(n_customers, n_service_units, t_mean_service, t_mean_customers):

    t_res = []
    blocked_frac_res = []
    departures_res =  []

    for _ in range(n_service_units):
        # State variables
        departure_times = []        # List of departure times for custormers currently being serviced 
        number_of_blocked = 0    
               
        t = 0                       
        t_arrival = 0               

        # Loop through all customers
        for _ in range(int(n_customers)):

            t += t_arrival # jumping to next event

            #Update departure list if not empty: keep only customers in list, whose departure time > current time
            if len(departure_times) != 0:
                departure_times = [dep for dep in departure_times if dep > t]

            #Check whether capacity is full. If full, block customer
            if len(departure_times) >= 10:
                number_of_blocked += 1                                     
            #If not full, assign service time and add customer to departure list
            else:
                #Assign service time for current customer
                t_service = np.random.exponential(t_mean_service)     
                #Add customers departure time to departure_times list
                departure_times.append(t+t_service)

            # Arrival time for next costumer
            t_arrival = np.random.exponential(t_mean_customers)
        
        # Storing results for every service unit
        t_res.append(t)
        blocked_frac_res.append(number_of_blocked/n_customers)
        departures_res.append(departure_times)

    return np.array(blocked_frac_res), np.array(t_res), departures_res

# Run simulation
blocked_frac_res, t_res, departures_res = block_system_poisson(
                                                                n_customers=1*1e4,
                                                                n_service_units=10, 
                                                                t_mean_service=8, 
                                                                t_mean_customers=1
                                                                )


print("Blocked fraction:")
print(blocked_frac_res)

mean_block = np.mean(blocked_frac_res)
std_block = np.std(blocked_frac_res, ddof=1)
n = len(blocked_frac_res)

ci_low = mean_block - 1.96 * std_block / np.sqrt(n)
ci_high = mean_block + 1.96 * std_block / np.sqrt(n)

print("Mean blocked fraction:", mean_block)
print("95% CI:", ci_low, ci_high)

print( "-------------- Control variate estimator --------------")
n_customers=1*1e4
n_service_units=10
t_mean_customers=1

# Crude estimator
mean_crude = np.mean(blocked_frac_res)
std_crude = np.std(blocked_frac_res, ddof=1)
ci_crude = (
    mean_crude - 1.96 * std_crude / np.sqrt(n_service_units),
    mean_crude + 1.96 * std_crude / np.sqrt(n_service_units)
)

print("Crude estimator:", mean_crude)
print("Crude 95% CI:", ci_crude)

# Control variate: total arrival time T
E_T = n_customers * t_mean_customers  # known expectation

cov_BT = np.cov(blocked_frac_res, t_res, ddof=1)[0, 1]
var_T = np.var(t_res, ddof=1)

c = -cov_BT / var_T
print("Control variate coefficient c:", c)

# Adjusted observations
Y = blocked_frac_res + c * (t_res - E_T)

# Control variate estimator
mean_cv = np.mean(Y)
std_cv = np.std(Y, ddof=1)
ci_cv = (
    mean_cv - 1.96 * std_cv / np.sqrt(n_service_units),
    mean_cv + 1.96 * std_cv / np.sqrt(n_service_units)
)

print("Control variate estimator:", mean_cv)
print("Control variate 95% CI:", ci_cv)


Blocked fraction:
[0.1391 0.1255 0.1171 0.1146 0.1164 0.128  0.1255 0.1172 0.1179 0.1153]
Mean blocked fraction: 0.12165999999999999
95% CI: 0.11682964372328498 0.12649035627671498
-------------- Control variate estimator --------------
Crude estimator: 0.12165999999999999
Crude 95% CI: (0.11682964372328498, 0.12649035627671498)
Control variate coefficient c: 4.9669155615138134e-05
Control variate estimator: 0.12235964525356127
Control variate 95% CI: (0.1193777096051084, 0.12534158090201414)


In [ ]:
# Exercise 5 - Part 7

import numpy as np
from scipy.stats import norm

# Crude Monte Carlo estimator for P(Z > a)
def crude_mc(a, n):
    Z = np.random.randn(n)  # Z ~ N(0,1)
    h = (Z > a).astype(float)
    theta_hat = np.mean(h)
    var_hat = np.var(h, ddof=1)
    return theta_hat, var_hat


# Importance Sampling estimator
# Using g = Normal(mean=a, variance=sigma^2)
def importance_sampling(a, sigma, n):
    # Sample from g
    Y = np.random.normal(loc=a, scale=sigma, size=n)

    # h(Y) = 1{Y > a}
    h = (Y > a).astype(float)

    # f(Y) = standard normal density
    f = norm.pdf(Y, loc=0, scale=1)

    # g(Y) = N(a, sigma^2) density
    g = norm.pdf(Y, loc=a, scale=sigma)

    # IS weights
    w = f / g

    # IS estimator
    theta_hat = np.mean(h * w)
    var_hat = np.var(h * w, ddof=1)

    return theta_hat, var_hat


# Run experiments

a_values = [1, 2, 4]
n_values = [10_000, 100_000]
sigma = 1  # start with sigma^2 = 1

for a in a_values:
    print(f"-----------------Estimating P(Z > {a})-----------------")

    true_value = 1 - norm.cdf(a)
    print(f"True value: {true_value:.6f}")

    for n in n_values:
        print(f"\nSample size n = {n}")

        # Crude MC
        crude_est, crude_var = crude_mc(a, n)
        print(f"Crude MC estimate: {crude_est:.6f}")
        print(f"Crude MC variance: {crude_var:.6e}")

        # Importance Sampling
        is_est, is_var = importance_sampling(a, sigma, n)
        print(f"IS estimate: {is_est:.6f}")
        print(f"IS variance: {is_var:.6e}")

        # Efficiency comparison
        efficiency = crude_var / is_var
        print(f"Efficiency gain (crude_var / is_var): {efficiency:.2f}x")



-----------------Estimating P(Z > 1)-----------------
True value: 0.158655

Sample size n = 10000
Crude MC estimate: 0.161100
Crude MC variance: 1.351603e-01
IS estimate: 0.158587
IS variance: 3.639638e-02
Efficiency gain (crude_var / is_var): 3.71x

Sample size n = 100000
Crude MC estimate: 0.158800
Crude MC variance: 1.335839e-01
IS estimate: 0.158784
IS variance: 3.657991e-02
Efficiency gain (crude_var / is_var): 3.65x
-----------------Estimating P(Z > 2)-----------------
True value: 0.022750

Sample size n = 10000
Crude MC estimate: 0.021700
Crude MC variance: 2.123123e-02
IS estimate: 0.022368
IS variance: 1.190162e-03
Efficiency gain (crude_var / is_var): 17.84x

Sample size n = 100000
Crude MC estimate: 0.022740
Crude MC variance: 2.222311e-02
IS estimate: 0.022756
IS variance: 1.211661e-03
Efficiency gain (crude_var / is_var): 18.34x
-----------------Estimating P(Z > 4)-----------------
True value: 0.000032

Sample size n = 10000
Crude MC estimate: 0.000000
Crude MC variance: 0

Observation: Crude Monte Carlo performs poorly for rare events such as P(Z>4), because very few samples fall in the tail. Importance sampling using a shifted normal distribution dramatically reduces variance by sampling directly in the region that contributes most to the probability.

In [ ]:
# Exercise 5 - Part 8

Use importance sampling with 

\begin{equation}
g(x)=\lambda e^{-\lambda x}
\end{equation}

to estimate

\begin{equation}
\int_0^1 e^x \: dx
\end{equation}

In [34]:
import numpy as np

true_value = np.e - 1

# Importance sampling estimator using g(x)
def importance_sampling(lambda_, n):
    # Sample from exponential distribution
    Y = np.random.exponential(scale=1/lambda_, size=n)

    # Keep only samples inside [0,1]
    mask = (Y <= 1)
    Y = Y[mask]

    # h(x) = e^x, f(x) = 1 on [0,1]
    h = np.exp(Y)

    g = lambda_ * np.exp(-lambda_ * Y)

    # IS weights
    w = h / g

    # Estimator
    theta_hat = np.mean(w)
    var_hat = np.var(w, ddof=1)

    return theta_hat, var_hat, len(Y)


# Experiment with different lambda values
lambdas = [0.1, 0.5, 1, 2, 4, 8]
n = 100000

print("True value:", true_value)

results = []

for lam in lambdas:
    est, var, used = importance_sampling(lam, n)
    results.append((lam, est, var, used))
    print(f"\nlambda = {lam}")
    print(f"Estimate: {est:.6f}")
    print(f"Variance: {var:.6e}")
    print(f"Samples inside [0,1]: {used}")


# Identify optimal lambda (minimum variance)
best = min(results, key=lambda x: x[2])
print("-------------------")
print("Optimal lambda (empirical):", best[0])
print("Variance at optimal lambda:", best[2])
print("-------------------")


True value: 1.718281828459045

lambda = 0.1
Estimate: 17.988040
Variance: 3.242487e+01
Samples inside [0,1]: 9564

lambda = 0.5
Estimate: 4.372077
Variance: 3.669605e+00
Samples inside [0,1]: 38971

lambda = 1
Estimate: 2.716448
Variance: 2.674603e+00
Samples inside [0,1]: 63288

lambda = 2
Estimate: 1.984951
Variance: 3.781881e+00
Samples inside [0,1]: 86415

lambda = 4
Estimate: 1.746092
Variance: 1.391304e+01
Samples inside [0,1]: 98128

lambda = 8
Estimate: 1.709312
Variance: 2.723304e+02
Samples inside [0,1]: 99963
-------------------
Optimal lambda (empirical): 1
Variance at optimal lambda: 2.674602500900141
-------------------


In [ ]:
# Exercise 5 - Part 9

Pareto importance sampling using the first--moment distribution

$X$ follows a Pareto distribution with density


$
f(x)=\frac{\alpha x_m^\alpha}{x^{\alpha+1}}, \qquad x\ge x_m,
$


and mean


$
\mu=\mathbb{E}[X]=\frac{\alpha x_m}{\alpha-1}.
$



To estimate $\mu$ using importance sampling, we choose the first-moment distribution


$
g(x)=\frac{x f(x)}{\mu}.
$


The IS estimator is


$
\hat{\mu}_{IS}
=\frac{1}{n}\sum_{i=1}^n X_i \frac{f(X_i)}{g(X_i)}.
$


Since

$
\frac{f(x)}{g(x)}=\frac{\mu}{x},
$


each summand becomes

$
X_i \frac{f(X_i)}{g(X_i)}=\mu,
$

so


$
\hat{\mu}_{IS}=\mu.
$



Thus the estimator has zero variance. but this approach is not meaningful in practice, because defining $g$ requires knowing $\mu$ in advance.

More generally, for estimating $\theta=\mathbb{E}[h(X)]$, the zero--variance IS density is


$
g^*(x)=\frac{|h(x)|f(x)}{\theta},
$


which again depends on the unknown quantity.

The optimal $g$ is proportional to $h(x)f(x)$. For the Pareto mean, $h(x)=x$, so $g(x)\propto x f(x)\propto x^{-\alpha}$. In practice, this suggests choosing a proposal distribution with a heavier tail (e.g. a Pareto with a smaller shape parameter) to approximate the ideal first-moment distribution and reduce variance.
